# Gold Layer

## Objetivo

La capa Gold tiene como propósito transformar los datos limpios de Silver en un modelo analítico optimizado para consumo de negocio.


# Diseño del Modelo

Se implementará un Star Schema compuesto por:

## Fact Table

fact_port_traffic

Medida principal:
- port_traffic_teu

## Dimensions

dim_country
dim_date
dim_indicator

Este modelo permite optimizar consultas analíticas y facilita el consumo desde herramientas BI.

# Lectura de Silver

Se utiliza la tabla Silver como fuente para construir el modelo dimensional Gold.

In [0]:
df_silver = spark.table(
    "workspace.silver.port_traffic_clean"
)

display(df_silver.limit(10))

country_id,country_name,country_iso3,indicator_id,indicator_name,year,port_traffic_teu,record_hash,silver_load_timestamp
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2023,8917517.0,8bf63ccaca93829f850b9d6bdc9c5665,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2022,8186999.0,d8733e84e9367373c421d3dd7cf7f652,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2021,8567214.0,c6fd8daeaffd45e9fb5bc19dd9007f59,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2020,8100808.0,e596a01c5258932dbf1cf87823e20b1d,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2019,8562750.13,ac0b490f4c3e5c8ce2e5117060022e1e,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2018,7919906.59,9da4c2d5d3e9994215c9622151ec6f96,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2017,7328156.16,afa86e16d9ca7b99fd8815b333c7f904,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2016,6464269.34,13992a9eea5a15fc9d5813bf5b200b48,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2015,6722792.36,e3cd1079bc7da38b8847a705d2defa81,2026-09-22T17:18:31.721Z
ZI,Africa Western and Central,AFW,IS.SHP.GOOD.TU,Container port traffic (TEU: 20 foot equivalent units),2014,6567432.71,57eeab2aeff37270c394534f784fb6d3,2026-09-22T17:18:31.721Z



# Filtrado de entidades analíticas

Durante el análisis Silver se identificó que la API del Banco Mundial contiene países y diferentes agregaciones económicas y regionales en el campo "country".

Para el modelo Gold se utilizarán únicamente países con códigos ISO válidos.


In [0]:
from pyspark.sql.functions import col, length

df_gold_base = (

    df_silver

    .filter(
        length(col("country_iso3")) == 3
    )

    .filter(
        col("country_iso3").isNotNull()
    )

)

# Validaciones

Antes de crear las dimensiones se validará la siguiente información:

- Cantidad de países
- Cobertura temporal
- Volumen de registros

In [0]:
# Cantidad de registros
df_gold_base.count()

2748

In [0]:
# Cantidad de países
df_gold_base.select(
    "country_iso3",
    "country_name"
).distinct().count()

206

In [0]:
# Cobertura temporal
from pyspark.sql.functions import min,max

df_gold_base.select(
    min("year"),
    max("year")
).show()

+---------+---------+
|min(year)|max(year)|
+---------+---------+
|     2005|     2024|
+---------+---------+



Los resultados obtenidos de las validaciones muestran:
- Cantidad de países: 206
- Cobertura temporal: 2005 - 2024
- Volumen de registros: 2748


## Construcción de dimensiones



# Dimension Country

Contiene la información descriptiva de cada país.

In [0]:
# Creamos la dimensión country
dim_country = (

    df_gold_base

    .select(

        "country_id",

        "country_iso3",

        "country_name"

    )

    .dropDuplicates()

)

In [0]:
# Guardamos la dimensión country en delta y el workspace gold

(
    dim_country.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_country"
    )
)            

# Dimension Date

Contiene los años disponibles para análisis histórico.

In [0]:
# Creamos la dimensión Date
dim_date = (

    df_gold_base

    .select("year")

    .dropDuplicates()

    .orderBy("year")

)

In [0]:
# Guardamos la dimensión Date en delta
(
    dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_date"
    )
)

# Dimension Indicator

Contiene los indicadores disponibles en el modelo.

In [0]:
# Creamos la dimensión indicator            
dim_indicator = (

    df_gold_base

    .select(

        "indicator_id",

        "indicator_name"

    )

    .dropDuplicates()

)

In [0]:
# Guardamos en delta la dimensión indicator
(
    dim_indicator.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.dim_indicator"
    )
)

# Fact Table

Contiene las métricas de tráfico portuario.

Medida:

- port_traffic_teu

In [0]:
# Creamos la tabla de hechos

fact_port_traffic = (

    df_gold_base

    .select(

        "country_id",

        "indicator_id",

        "year",

        "port_traffic_teu",

        "record_hash"

    )

)

In [0]:
# Guardamos la tabla de hechos en delta

(
    fact_port_traffic.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "workspace.gold.fact_port_traffic"
    )
)


# Validación Final

Verificación de tablas creadas en Gold.

In [0]:
spark.sql("""
SHOW TABLES IN workspace.gold
""").show(truncate=False)

+--------+-----------------+-----------+
|database|tableName        |isTemporary|
+--------+-----------------+-----------+
|gold    |dim_country      |false      |
|gold    |dim_date         |false      |
|gold    |dim_indicator    |false      |
|gold    |fact_port_traffic|false      |
+--------+-----------------+-----------+



# Resultados y resumen de capa Gold

Se obtuvo la siguietne Cobertura analítica:

- 206 países disponibles para análisis.
- Cobertura histórica: 2005 - 2024.
- 2.748 registros analíticos listos para consumo por Power BI.
- Star Schema implementado correctamente.

La validación también confirma las 4 tablaas fueron creadas correctamente: 

- workspace.gold.dim_country
- workspace.gold.dim_date
- workspace.gold.dim_indicator
- workspace.gold.fact_port_traffic